In [ ]:
# Step 0: Install PyTabKit
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pytabkit"])
print("PyTabKit installed successfully.")


In [ ]:
# Step 1: Imports and Device Setup
import os
import gc
import glob
import time
import random
import warnings
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder

import torch
from pytabkit import TabM_D_Classifier

warnings.filterwarnings('ignore')

print("PyTorch version: ", torch.__version__)
print("CUDA available:  ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device:      ", torch.cuda.get_device_name(0))
    print("Device count:    ", torch.cuda.device_count())
print("PyTabKit version:", version("pytabkit"))

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)


In [ ]:
# Step 2: Dynamic Dataset Discovery & Loading
train_candidates = (
    glob.glob('/kaggle/input/**/train.csv', recursive=True)
    + glob.glob('**/train.csv', recursive=True)
    + glob.glob('../../dataset/train.csv')
    + glob.glob('pgs-s6e11/dataset/train.csv')
)

if not train_candidates:
    raise FileNotFoundError("Could not find train.csv in any candidate paths.")

TRAIN_PATH = Path(train_candidates[0])
DATA_DIR = TRAIN_PATH.parent
TEST_PATH = DATA_DIR / "test.csv"
SUB_PATH = DATA_DIR / "sample_submission.csv"

print(f"Loading data from: {DATA_DIR}")
t0 = time.time()
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SUB_PATH)

print(f"Loaded train: {train_df.shape}, test: {test_df.shape} in {time.time() - t0:.2f}s")

ID = 'id'
TARGET = 'is_fraud'

train_id = train_df[ID].values
test_id = test_df[ID].values
y = train_df[TARGET].values.astype(int)

X = train_df.drop(columns=[ID, TARGET]).copy()
X_test = test_df.drop(columns=[ID]).copy()
del train_df, test_df
gc.collect()

print(f"Fraud rate: {y.mean():.4%} ({y.sum():,} positive frauds out of {len(y):,} rows)")


In [ ]:
# Step 3: Domain Feature Engineering
t0 = time.time()

cat_cols = X.select_dtypes(include=['object', 'bool']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object', 'bool']).columns.tolist()
print(f"Base columns: {len(cat_cols)} categorical, {len(num_cols)} numerical")

important_combos = [
    ('merchant_category', 'card_type', 'txn_count_last_24h'),
    ('merchant_category', 'customer_age'),
]

category_map = {}

def feature_engineering(df, fit=False):
    # 1. Decimal / Fractional parts
    for col in ['amount_usd', 'hours_since_last_txn', 'velocity_score', 'distance_from_home_km']:
        if col in df.columns:
            df[f'_{col}_decimal'] = (df[col] % 1).round(2).astype('float32')

    # 2. Benford First Digit of amount & Cent Indicators
    if 'amount_usd' in df.columns:
        df['_amount_first_digit'] = df['amount_usd'].astype(str).str.extract(r'([1-9])')[0].fillna('0').astype(int).astype('category')
        df['_is_round_dollar'] = ((df['amount_usd'] % 1).round(2) == 0).astype(int).astype('category')
        df['_is_99_cents'] = ((df['amount_usd'] % 1).round(2) == 0.99).astype(int).astype('category')

    # 3. Discretize continuous distributions into integer category codes
    for col in num_cols + ['_amount_usd_decimal', '_hours_since_last_txn_decimal', '_velocity_score_decimal', '_distance_from_home_km_decimal']:
        if col in df.columns:
            cat_name = f'{col}_cat_' if col in num_cols else f'{col[1:]}_cat_'
            values = np.floor(df[col]) if col in num_cols else df[col]
            if fit:
                codes, uniques = values.factorize()
                category_map[col] = uniques
            else:
                uniques = category_map[col]
                code_map = {cat: i for i, cat in enumerate(uniques)}
                codes = values.map(code_map).fillna(-1).astype('int32')
            df[cat_name] = codes
            df[cat_name] = df[cat_name].astype('category')

    # 4. Quantile Bins
    bin_config = {'merchant_risk_score': [7], 'hours_since_last_txn': [500, 1000]}
    for col, bins_list in bin_config.items():
        if col in df.columns:
            for n_bins in bins_list:
                bin_name = f'{col}_{n_bins}_quantile_bin_'
                if fit:
                    kb = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='quantile', subsample=None)
                    binned = kb.fit_transform(df[[col]]).ravel().astype('int32')
                    category_map[bin_name] = kb
                else:
                    kb = category_map[bin_name]
                    binned = kb.transform(df[[col]]).ravel().astype('int32')
                df[bin_name] = binned
                df[bin_name] = df[bin_name].astype('category')

    # 5. High-order interaction categories
    combo_names = []
    for cols in important_combos:
        combo_name = '_'.join(cols) + '_'
        combo_names.append(combo_name)
        combo_series = df[cols[0]].astype(str)
        for col in cols[1:]: 
            combo_series = combo_series + '_' + df[col].astype(str)
        if fit:
            codes, uniques = pd.factorize(combo_series, sort=False)
            category_map[combo_name] = uniques
        else:
            uniques = category_map[combo_name]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = combo_series.map(code_map).fillna(-1).astype('int32')
        df[combo_name] = codes
        df[combo_name] = df[combo_name].astype('category')

    new_cat_cols = [col for col in df.columns if col.endswith('_')]
    new_num_cols = [col for col in df.columns if col.startswith('_')]
    return df, new_cat_cols, new_num_cols, combo_names

X, new_cat_cols, new_num_cols, combo_names = feature_engineering(X, fit=True)
X_test, _, _, _ = feature_engineering(X_test, fit=False)
cat_cols += new_cat_cols
num_cols += new_num_cols
print(f"Feature engineering completed in {time.time() - t0:.2f}s: X shape={X.shape}, X_test={X_test.shape}")


In [ ]:
# Step 4: 5-Fold Stratified TabM Training Loop
FOLDS = 5
SEED = 42

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
oof_preds = np.zeros(len(X), dtype=np.float64)
test_preds = np.zeros(len(X_test), dtype=np.float64)

tabm_params = {
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'random_state': SEED,
    'val_metric_name': 'cross_entropy',
    'tabm_k': 16,
    'batch_size': 1024,
    'lr': 0.002,
    'weight_decay': 0.01,
    'n_epochs': 15,
    'patience': 5,
    'allow_amp': True,
    'verbosity': 1,
}

te_cols = combo_names + ['hours_since_last_txn']

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    fold_t0 = time.time()
    print("=" * 60)
    print(f">>> Fold {fold}/{FOLDS} TabM Deep Ensemble Training")
    print("=" * 60)

    X_tr = X.iloc[tr_idx].copy()
    X_val = X.iloc[val_idx].copy()
    y_tr = y[tr_idx]
    y_val = y[val_idx]
    X_tst = X_test.copy()

    # Out-of-fold Target Encoding on key interactions
    TE = TargetEncoder(cv=FOLDS, smooth='auto', shuffle=True, random_state=SEED + fold)
    tr_enc = TE.fit_transform(X_tr[te_cols], y_tr)
    val_enc = TE.transform(X_val[te_cols])
    tst_enc = TE.transform(X_tst[te_cols])

    te_names = [f'_{col}_TE' for col in te_cols]
    X_tr[te_names] = tr_enc.astype('float32')
    X_val[te_names] = val_enc.astype('float32')
    X_tst[te_names] = tst_enc.astype('float32')

    print(f"Features for training: {X_tr.shape[1]} columns")

    fold_params = dict(tabm_params)
    fold_params['random_state'] = SEED + fold

    model = TabM_D_Classifier(**fold_params)
    model.fit(X_tr, y_tr, X_val, y_val)

    val_preds = model.predict_proba(X_val)[:, 1]
    fold_test_preds = model.predict_proba(X_tst)[:, 1]

    oof_preds[val_idx] = val_preds
    test_preds += fold_test_preds / FOLDS

    fold_auc = roc_auc_score(y_val, val_preds)
    fold_prauc = average_precision_score(y_val, val_preds)
    elapsed = time.time() - fold_t0
    print(f"Fold {fold} finished in {elapsed:.1f}s | ROC-AUC: {fold_auc:.6f} | PR-AUC: {fold_prauc:.6f}")

    del X_tr, X_val, X_tst, y_tr, y_val, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
# Step 5: Overall Evaluation & Export
overall_auc = roc_auc_score(y, oof_preds)
overall_prauc = average_precision_score(y, oof_preds)

print("=" * 60)
print(f">>> Overall TabM 5-Fold OOF ROC-AUC: {overall_auc:.6f}")
print(f">>> Overall TabM 5-Fold OOF PR-AUC:  {overall_prauc:.6f}")
print("=" * 60)

np.save('oof_tabm.npy', oof_preds.astype(np.float32))
np.save('test_tabm.npy', test_preds.astype(np.float32))

r_test = (rankdata(test_preds) - 0.5) / len(test_preds)
sub = pd.DataFrame({'id': test_id, 'is_fraud': r_test})
sub.to_csv('tabm_submission.csv', index=False)

print("Exported files:")
print(f"  oof_tabm.npy:        {os.path.getsize('oof_tabm.npy') / (1024*1024):.2f} MB")
print(f"  test_tabm.npy:       {os.path.getsize('test_tabm.npy') / (1024*1024):.2f} MB")
print(f"  tabm_submission.csv: {os.path.getsize('tabm_submission.csv') / (1024*1024):.2f} MB")
print("All tasks finished successfully.")
